# Task 1 (part 2): Multimodal LLM Description via Ollama -- Colab Version

Runs entirely inside this Colab runtime, including a fresh Ollama server --
this sidesteps a persistent local Mac install bug (`llama3.2-vision` fails
to load with `unknown model architecture: 'mllama'` despite a full model
re-download and app reinstall). Setup cells below are the exact working
pattern from `lab5_hybrid_unet_llm_colab.ipynb`, adapted for Task 1's
prompt comparison instead of the full hybrid pipeline.

**Steps:** run every cell top to bottom. You'll be asked to upload one
image (`train_062.png`, the representative training image) partway
through. At the end, two `.txt` files download automatically -- send both
back.

## Setup 1 of 3: install the Ollama server + Python client

**Important:** pinned to `OLLAMA_VERSION=0.22.1` deliberately (a
confirmed real stable release, April 2026). As of Ollama v0.30.0 (May
2026), the `mllama` architecture that `llama3.2-vision` needs was dropped
from Ollama's new inference engine and has not been restored (confirmed
via multiple open upstream issues, e.g. ollama/ollama#16547, #16490 --
`error loading model: unknown model architecture: 'mllama'`, reproduced
identically on a fresh Mac install and a fresh Colab install, ruling out
anything machine-specific). Every version before 0.30.0 supports it fine,
so we install an older, confirmed-to-exist release here specifically to
get a working vision model, rather than "latest".

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=0.22.1 sh
!pip -q install ollama
!which ollama && ollama --version || echo "INSTALL FAILED -- ollama binary not found. Stop here and report this."


## Setup 2 of 3: start the Ollama server in the background

In [ ]:
import os, subprocess, time, requests

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"

def ollama_up():
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if not ollama_up():
    _log = open("ollama_serve.log", "w")
    subprocess.Popen(["ollama", "serve"], stdout=_log, stderr=_log)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(1)

print("Ollama server reachable:", ollama_up())

## Setup 3 of 3: pull the vision model

This downloads ~7-8GB the first time -- takes a few minutes on Colab's connection.

In [ ]:
!ollama pull llama3.2-vision
!ollama list

## Upload the representative image

Upload `train_062.png` (27 nuclei, "normal" density, closest to the median
for that regime -- the same justified, non-arbitrary pick used in the
local pipeline).

In [ ]:
from google.colab import files

print("Please upload train_062.png (from data/nuclei_dataset/train/images/ in your project folder):")
uploaded = files.upload()
IMAGE_PATH = list(uploaded.keys())[0]
print(f"\nUsing image: {IMAGE_PATH}")

## Task 1 pipeline code

Same logic as `src/imaging_pipeline/vlm_description.py` in the main
project repo, inlined here since this notebook is self-contained (no repo
clone needed for a single-image test).

In [ ]:
import json
from ollama import chat

VLM_MODEL = "llama3.2-vision"

NAIVE_PROMPT = "What is in this image?"


def build_optimised_prompt() -> str:
    return (
        "You are a scientific image-cataloguing assistant helping annotate "
        "microscopy images for a research dataset.\n\n"
        "You are NOT a diagnostic tool. Do not provide any clinical, "
        "diagnostic, or disease-related interpretation, even if asked. "
        "Describe only what is visually present, in purely observational "
        "terms (shapes, brightness, spatial arrangement, apparent density, "
        "texture).\n\n"
        "Respond with a JSON object with EXACTLY these four fields, and "
        "nothing else:\n"
        "{\n"
        '  "modality": "<imaging modality if visually identifiable, else '
        '\\"uncertain\\">",\n'
        '  "tissue_type": "<general sample/tissue type if identifiable, '
        'else \\"uncertain\\">",\n'
        '  "notable_features": "<one short factual sentence on visually '
        'notable features>",\n'
        '  "image_quality": "<one of: good, moderate, poor>"\n'
        "}\n\n"
        "If you are not confident about a field, write \"uncertain\" for "
        "that field rather than guessing. Respond with ONLY the JSON "
        "object -- no preamble, no explanation, no markdown code fences."
    )


def query_vlm(image_path, prompt, model=VLM_MODEL, temperature=None):
    options = {}
    if temperature is not None:
        options["temperature"] = temperature
    response = chat(
        model=model,
        messages=[{"role": "user", "content": prompt, "images": [str(image_path)]}],
        options=options,
    )
    return response["message"]["content"]


def parse_json_response(text: str) -> dict:
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1] if "\n" in text else text
        if text.rstrip().endswith("```"):
            text = text.rstrip()[:-3]
        text = text.strip()
    start = text.find("{")
    end = text.rfind("}")
    candidate = text[start:end + 1] if (start != -1 and end != -1 and end > start) else text
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        return {"error": "could not parse JSON", "raw": text}


def has_required_fields(record, required=("modality", "tissue_type", "notable_features", "image_quality")):
    return all(field in record for field in required)


print("Pipeline functions defined.")

## Run the naive vs. optimised prompt comparison (temperature=0, reproducible)

In [ ]:
print("Running naive prompt...")
naive_response = query_vlm(IMAGE_PATH, NAIVE_PROMPT, temperature=0.0)
print("Done.\n")

print("Running optimised prompt...")
optimised_prompt = build_optimised_prompt()
optimised_response = query_vlm(IMAGE_PATH, optimised_prompt, temperature=0.0)
optimised_record = parse_json_response(optimised_response)
print("Done.\n")

print("=== NAIVE PROMPT RESPONSE ===")
print(naive_response)
print("\n=== OPTIMISED PROMPT RESPONSE (raw) ===")
print(optimised_response)
print("\n=== OPTIMISED PROMPT: PARSED RECORD ===")
print(optimised_record)
print(f"\nValid (has all 4 required fields): {has_required_fields(optimised_record)}")

In [ ]:
comparison_text = f"""Representative image: {IMAGE_PATH}
Model: {VLM_MODEL}

{"="*70}
NAIVE PROMPT
{"="*70}
{NAIVE_PROMPT}

--- Response ---
{naive_response}

{"="*70}
OPTIMISED PROMPT
{"="*70}
{optimised_prompt}

--- Raw response ---
{optimised_response}

--- Parsed JSON record ---
{optimised_record}

Valid (all 4 required fields present): {has_required_fields(optimised_record)}
"""

with open("task1_prompt_comparison.txt", "w") as f:
    f.write(comparison_text)

print("Saved task1_prompt_comparison.txt")

## Repeated-run variability check

Same prompt, same image, 3 calls at a non-zero temperature (0.8) --
deliberately NOT temperature=0, since the point here is to demonstrate
genuine run-to-run variability, not suppress it.

In [ ]:
print("Running 3 repeated calls at temperature=0.8...")
runs = [query_vlm(IMAGE_PATH, optimised_prompt, temperature=0.8) for _ in range(3)]
print("Done.\n")

all_identical = len(set(runs)) == 1
print(f"All 3 runs produced identical text: {all_identical}")
for i, r in enumerate(runs):
    print(f"\n--- Run {i+1} ---")
    print(r)

In [ ]:
runs_text = f"""Repeated-run variability check
Image: {IMAGE_PATH}, model: {VLM_MODEL}, temperature=0.8, n_runs={len(runs)}

All runs identical: {all_identical}

"""
for i, r in enumerate(runs):
    runs_text += f"--- Run {i+1} ---\n{r}\n\n"

with open("task1_repeated_runs.txt", "w") as f:
    f.write(runs_text)

print("Saved task1_repeated_runs.txt")

## Download both output files

In [ ]:
from google.colab import files

files.download("task1_prompt_comparison.txt")
files.download("task1_repeated_runs.txt")

print("Both files downloading -- send both back once you have them.")